# Case Linking — silver_case_link_groups Tests

Independently verifies that `silver_case_link_groups` is consistent with its three source bronze tables:
- `stg_segmentation_states` — the set of ARIA cases in scope (Active Case Segmentation)
- `bronze_appealcase_caseappellant_appellant` — supplies `BirthDate` per case
- `bronze_appealcase_link_linkdetail` — supplies `LinkNo` and `ReasonLinkId`

Approach: check each aspect that built the silver table to verify each a different to way to dev, which uses joins
Checking:
 - every active case appears in silver, and nothing extra has crept in
 - each case's birthdate in silver matches the birthdate in the appellant table
 - every link in silver actually exists in the source link table
 - the total row count adds up to what you'd expect from the source data

If each of these are correct then the table will pass all checks

In [0]:
########################
# test Setup
########################

state_under_test = "caseLinking"
run_tag_default = "silver_case_link_groups"

dbutils.widgets.text("run_tag", run_tag_default)
run_tag = dbutils.widgets.get("run_tag") or run_tag_default

In [0]:
import os
import sys
import time as perf_time
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.functions import col, count, countDistinct
import inspect

run_user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
test_results_path = "/Workspace/Users/" + run_user + "/Results/Case_Linking_Tests"
os.makedirs(test_results_path, exist_ok=True)

for _mod in list(sys.modules.keys()):
    if _mod.startswith("Test_Functions") or _mod == "models.test_result":
        del sys.modules[_mod]

from models.test_result import TestResult
from Test_Functions.report_helpers import (
    build_report_folder,
    write_csv, write_xlsx, write_html,
    count_by_status,
    create_run, create_results,
)

print(f"User: {run_user}")
print(f"state_under_test: {state_under_test}")
print(f"run_tag: {run_tag}")
print(f"Results folder root: {test_results_path}")

In [0]:
#######################
# Spark Config (state-independent — runs once)
#######################
config_path = "dbfs:/configs/config.json"
config = spark.read.option("multiline", "true").json(config_path)
first_row = config.first()
env = first_row["env"].strip().lower()
env_name = env
lz_key = first_row["lz_key"].strip().lower()
keyvault_name = f"ingest{lz_key}-meta002-{env}"
KeyVault_name = keyvault_name

client_secret = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-CLIENT-SECRET")
tenant_id     = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-TENANT-ID")
client_id     = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-CLIENT-ID")

curated_storage    = f"ingest{lz_key}curated{env}"
checkpoint_storage = f"ingest{lz_key}xcutting{env}"
raw_storage        = f"ingest{lz_key}raw{env}"
landing_storage    = f"ingest{lz_key}landing{env}"
external_storage   = f"ingest{lz_key}external{env}"

for storage in (curated_storage, checkpoint_storage, raw_storage, landing_storage, external_storage):
    spark.conf.set(f"fs.azure.account.auth.type.{storage}.dfs.core.windows.net", "OAuth")
    spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage}.dfs.core.windows.net", client_id)
    spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage}.dfs.core.windows.net", client_secret)
    spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

print(f"env={env}  lz_key={lz_key}")

## Load tables

In [0]:
silver_case_link_groups = spark.table("hive_metastore.ariadm_active_appeals.silver_case_link_groups")
src_segmentation        = spark.table("hive_metastore.ariadm_active_appeals.stg_segmentation_states")
src_appellant           = spark.table("hive_metastore.ariadm_active_appeals.bronze_appealcase_caseappellant_appellant")
src_link                = spark.table("hive_metastore.ariadm_active_appeals.bronze_appealcase_link_linkdetail")

print(f"silver_case_link_groups rows: {silver_case_link_groups.count():,}")
print(f"src_segmentation rows:        {src_segmentation.count():,}")
print(f"src_appellant rows:           {src_appellant.count():,}")
print(f"src_link rows:                {src_link.count():,}")

In [0]:
run_start_datetime = datetime.utcnow()
all_test_results = []
t0 = perf_time.time()

# Coverage — every segmentation case appears in silver, and nothing extra

In [0]:
def test_silver_case_coverage(silver_df, seg_df):
    fn = inspect.stack()[0].function
    results = []
    try:
        silver_cases = silver_df.select("CaseNo").distinct()
        seg_cases    = seg_df.select("CaseNo").distinct()

        seg_n    = seg_cases.count()
        silver_n = silver_cases.count()

        missing = seg_cases.subtract(silver_cases)
        extra   = silver_cases.subtract(seg_cases)
        miss_n  = missing.count()
        extra_n = extra.count()

        # 1) Every segmentation case must appear in silver
        msg = f"expected: every segmentation case in silver | actual: {miss_n} missing from silver (segmentation distinct={seg_n}, silver distinct={silver_n})"
        if miss_n:
            sample = [r.CaseNo for r in missing.limit(5).collect()]
            msg += f" (sample: {sample})"
        results.append(TestResult(
            "silver_case_link_groups.coverage_no_missing",
            "PASS" if miss_n == 0 else "FAIL", msg, state_under_test, fn))

        # 2) Silver should not introduce cases not in segmentation
        msg = f"expected: no cases in silver outside segmentation | actual: {extra_n} extra"
        if extra_n:
            sample = [r.CaseNo for r in extra.limit(5).collect()]
            msg += f" (sample: {sample})"
        results.append(TestResult(
            "silver_case_link_groups.coverage_no_extras",
            "PASS" if extra_n == 0 else "FAIL", msg, state_under_test, fn))
    except Exception as e:
        results.append(TestResult(
            "silver_case_link_groups.coverage", "FAIL", f"Test crashed: {e}",
            state_under_test, fn))
    return results


all_test_results.extend(test_silver_case_coverage(silver_case_link_groups, src_segmentation))

# BirthDate accuracy - silver.BirthDate must match the PRIMARY appellant (Relationship IS NULL) for that case

In [0]:
def test_silver_birthdate_matches_appellant(silver_df, appellant_df):
    # The primary appellant on a case has Relationship IS NULL and carries the BirthDate.
    # Dependants/descendants have a populated Relationship (and typically a null BirthDate).
    # silver should hold the primary's BirthDate, so compare against the primary row only -
    # otherwise the descendant rows produce false mismatches on any multi-appellant case.
    fn = inspect.stack()[0].function
    results = []
    try:
        primary_appellant = appellant_df.filter(col("Relationship").isNull())

        silver_pairs    = silver_df.select("CaseNo", "BirthDate").distinct()
        appellant_pairs = primary_appellant.select("CaseNo", "BirthDate").distinct()

        joined = (
            silver_pairs.alias("s")
                .join(appellant_pairs.alias("a"), "CaseNo", "left")
                .select(
                    col("CaseNo").alias("CaseNo"),
                    col("s.BirthDate").alias("silver_bd"),
                    col("a.BirthDate").alias("primary_bd"),
                )
        )

        mismatched = joined.filter(
            col("primary_bd").isNotNull() &
            ((col("silver_bd") != col("primary_bd")) |
             (col("silver_bd").isNull() & col("primary_bd").isNotNull()))
        )
        no_primary = joined.filter(col("primary_bd").isNull() & col("silver_bd").isNotNull())

        mm_n = mismatched.count()
        np_n = no_primary.count()

        ok = mm_n == 0
        msg = f"expected: silver.BirthDate == primary appellant BirthDate (Relationship IS NULL) | actual: {mm_n} mismatches"
        if mm_n:
            sample = [(r.CaseNo, r.silver_bd, r.primary_bd)
                      for r in mismatched.limit(5).collect()]
            msg += f" (sample CaseNo,silver,primary: {sample})"
        results.append(TestResult(
            "silver_case_link_groups.birthdate_matches_appellant",
            "PASS" if ok else "FAIL", msg, state_under_test, fn))

        msg = f"informational: {np_n} silver cases have a BirthDate but no primary appellant row (Relationship IS NULL)"
        if np_n:
            sample = [(r.CaseNo, r.silver_bd) for r in no_primary.limit(5).collect()]
            msg += f" (sample CaseNo,silver_bd: {sample})"
        results.append(TestResult(
            "silver_case_link_groups.birthdate_orphan",
            "PASS" if np_n == 0 else "FAIL", msg, state_under_test, fn))
    except Exception as e:
        results.append(TestResult(
            "silver_case_link_groups.birthdate", "FAIL", f"Test crashed: {e}",
            state_under_test, fn))
    return results


all_test_results.extend(test_silver_birthdate_matches_appellant(silver_case_link_groups, src_appellant))

# Link accuracy — silver link rows must exist in linkdetail; silver null rows must not have a link

In [0]:
def test_silver_link_matches_linkdetail(silver_df, link_df, seg_df):
    fn = inspect.stack()[0].function
    results = []
    try:
        # 1) Every (CaseNo, LinkNo, ReasonLinkId) in silver where LinkNo is non-null
        #    must exist as a row in linkdetail.
        silver_links = (
            silver_df.filter(col("LinkNo").isNotNull())
                     .select("CaseNo", "LinkNo", "ReasonLinkId")
                     .distinct()
        )
        link_rows = link_df.select("CaseNo", "LinkNo", "ReasonLinkId").distinct()

        orphans = silver_links.subtract(link_rows)
        orphan_n = orphans.count()
        msg = f"expected: every silver link triple exists in linkdetail | actual: {orphan_n} orphans"
        if orphan_n:
            sample = [(r.CaseNo, r.LinkNo, r.ReasonLinkId) for r in orphans.limit(5).collect()]
            msg += f" (sample: {sample})"
        results.append(TestResult(
            "silver_case_link_groups.link_triples_in_source",
            "PASS" if orphan_n == 0 else "FAIL", msg, state_under_test, fn))

        # 2) Cases in SEGMENTATION that have a link in source must have a link row in silver.
        #    (Cases with links in source but outside segmentation are correctly excluded —
        #     silver is scoped to active cases only.)
        seg_cases     = seg_df.select("CaseNo").distinct()
        linked_in_src = link_df.select("CaseNo").distinct()
        linked_in_seg = seg_cases.join(linked_in_src, "CaseNo", "inner")
        silver_linked = silver_df.filter(col("LinkNo").isNotNull()).select("CaseNo").distinct()

        missing_links = linked_in_seg.subtract(silver_linked)
        ml_n = missing_links.count()
        msg = f"expected: every active case with a link in source has a link row in silver | actual: {ml_n} missing"
        if ml_n:
            sample = [r.CaseNo for r in missing_links.limit(5).collect()]
            msg += f" (sample: {sample})"
        results.append(TestResult(
            "silver_case_link_groups.linked_cases_covered",
            "PASS" if ml_n == 0 else "FAIL", msg, state_under_test, fn))

        # 3) For cases with no link in source, silver should have only NULL LinkNo rows.
        cases_with_no_link = (
            silver_df.select("CaseNo").distinct()
                     .subtract(linked_in_src)
        )
        with_link_anyway = (
            silver_df.filter(col("LinkNo").isNotNull())
                     .select("CaseNo").distinct()
                     .join(cases_with_no_link, "CaseNo", "inner")
        )
        wla_n = with_link_anyway.count()
        msg = f"expected: cases with no link in source have only NULL LinkNo in silver | actual: {wla_n} have non-null"
        results.append(TestResult(
            "silver_case_link_groups.unlinked_cases_have_null",
            "PASS" if wla_n == 0 else "FAIL", msg, state_under_test, fn))
    except Exception as e:
        results.append(TestResult(
            "silver_case_link_groups.link_accuracy", "FAIL", f"Test crashed: {e}",
            state_under_test, fn))
    return results


all_test_results.extend(test_silver_link_matches_linkdetail(silver_case_link_groups, src_link, src_segmentation))

# Row count — silver row count is max(appellant_rows, 1) * max(link_rows, 1)

In [0]:
def test_silver_row_count_explained(silver_df, seg_df, app_df, link_df):
    """The silver row count should equal, per segmentation case,
       max(appellant_rows, 1) * max(link_rows, 1)
    """
    fn = inspect.stack()[0].function
    results = []
    try:
        seg_cases = seg_df.select("CaseNo").distinct()

        app_per_case = (
            seg_cases.alias("s")
                .join(app_df.select("CaseNo").alias("a"), col("s.CaseNo") == col("a.CaseNo"), "left")
                .groupBy("s.CaseNo")
                .agg(count("a.CaseNo").alias("app_rows"))
        )
        link_per_case = (
            seg_cases.alias("s")
                .join(link_df.select("CaseNo").alias("l"), col("s.CaseNo") == col("l.CaseNo"), "left")
                .groupBy("s.CaseNo")
                .agg(count("l.CaseNo").alias("link_rows"))
        )

        per_case = (
            app_per_case.join(link_per_case, "CaseNo", "inner")
                .withColumn("app_factor", F.when(col("app_rows") == 0, F.lit(1)).otherwise(col("app_rows")))
                .withColumn("link_factor", F.when(col("link_rows") == 0, F.lit(1)).otherwise(col("link_rows")))
                .withColumn("expected_silver_rows", col("app_factor") * col("link_factor"))
        )

        expected_total = per_case.agg(F.sum("expected_silver_rows").alias("t")).first().t or 0
        silver_total = silver_df.count()

        ok = expected_total == silver_total
        msg = (f"expected: {expected_total} silver rows "
               f"(sum of max(app_rows,1) * max(link_rows,1) per segmentation case) | "
               f"actual: {silver_total}")
        if not ok:
            # Surface the cases driving the gap: top contributors to the cardinality
            top = per_case.filter("expected_silver_rows > 1") \
                          .orderBy(col("expected_silver_rows").desc()).limit(5).collect()
            sample = [(r.CaseNo, r.app_rows, r.link_rows, r.expected_silver_rows) for r in top]
            msg += f" (top contributors CaseNo,app,link,expected: {sample})"
        results.append(TestResult(
            "silver_case_link_groups.row_count",
            "PASS" if ok else "FAIL", msg, state_under_test, fn))
    except Exception as e:
        results.append(TestResult(
            "silver_case_link_groups.row_count", "FAIL", f"Test crashed: {e}",
            state_under_test, fn))
    return results


all_test_results.extend(test_silver_row_count_explained(silver_case_link_groups, src_segmentation, src_appellant, src_link))

In [0]:
elapsed = perf_time.time() - t0
counts = count_by_status(all_test_results)

print(f"Tests complete in {elapsed:.1f}s")
print(f"  Total : {counts['TOTAL']}")
print(f"  Pass  : {counts['PASS']}")
print(f"  Fail  : {counts['FAIL']}")
print(f"  NoData: {counts['NO_DATA']}")
print(f"  Error : {counts['ERROR']}")

## Write reports + audit

In [0]:
status_order = {"FAIL": 0, "ERROR": 1, "NO_DATA": 2, "PASS": 3}
all_test_results.sort(key=lambda r: (status_order.get(str(r.status).upper(), 4), str(r.test_field)))

folder, timestamp, _ = build_report_folder(test_results_path, f"{state_under_test}_{run_tag}")
print(f"Reports folder: {folder}")

try:
    csv_path  = write_csv(all_test_results, state_under_test, folder, timestamp)
    print(f"CSV  : {csv_path}")
except Exception as e:
    print(f"write_csv FAILED: {e}")

try:
    run_id = create_run(
        spark,
        run_user=run_user,
        run_start_datetime=run_start_datetime,
        run_by_automation_name="Case_Linking_Tests",
        run_tag=run_tag,
        run_status=("FAILED" if any(str(getattr(r, "status", "")).upper().strip() in ("FAIL", "ERROR") for r in all_test_results) else "PASSED"),
        state_under_test=state_under_test,
    )
    print(f"Created run_id={run_id}")
    n = create_results(spark, run_id, all_test_results)
    print(f"Wrote {n} result rows")
except Exception as e:
    print(f"audit write FAILED: {e}")

In [0]:
from pyspark.sql.functions import col
display(silver_case_link_groups.filter(col("CaseNo") == "HU/00454/2025"))
display(src_appellant.filter(col("CaseNo") == "HU/00454/2025"))
display(src_link.filter(col("CaseNo") == "HU/00454/2025"))
display(src_segmentation.filter(col("CaseNo") == "HU/00454/2025"))

In [0]:
# Cell 1 — load tables
from pyspark.sql.functions import col, count

silver    = spark.table("hive_metastore.ariadm_active_appeals.silver_case_link_groups")
appellant = spark.table("hive_metastore.ariadm_active_appeals.bronze_appealcase_caseappellant_appellant")
link      = spark.table("hive_metastore.ariadm_active_appeals.bronze_appealcase_link_linkdetail")

silver_dups = (
    silver
        .filter(col("LinkNo").isNotNull())
        .groupBy("CaseNo", "LinkNo", "ReasonLinkId")
        .agg(count("*").alias("silver_row_count"))
        .filter(col("silver_row_count") > 1)
        .orderBy(col("silver_row_count").desc(), "CaseNo")
)

print(f"Duplicate (CaseNo, LinkNo, ReasonLinkId) triples in silver: {silver_dups.count()}")
print(f"Distinct cases affected:                                    {silver_dups.select('CaseNo').distinct().count()}")
display(silver_dups)

In [0]:
affected_cases = silver_dups.select("CaseNo").distinct()

silver_affected = (
    silver
        .join(affected_cases, "CaseNo", "inner")
        .orderBy("CaseNo", "LinkNo", "BirthDate")
)
print(f"Silver rows for affected cases: {silver_affected.count()}")
display(silver_affected)

In [0]:
app_per_case = (
    appellant
        .join(affected_cases, "CaseNo", "inner")
        .groupBy("CaseNo")
        .agg(count("*").alias("appellant_rows"))
        .orderBy(col("appellant_rows").desc())
)
print("Appellant rows per affected case (high counts = bigger silver multiplication):")
display(app_per_case)

print("Actual appellant rows for affected cases:")
display(
    appellant
        .join(affected_cases, "CaseNo", "inner")
        .orderBy("CaseNo")
)


In [0]:
from pyspark.sql.functions import col, count
silver = spark.table("hive_metastore.ariadm_active_appeals.silver_case_link_groups")
dups = (silver.filter(col("LinkNo").isNotNull())
            .groupBy("CaseNo", "LinkNo", "ReasonLinkId")
            .agg(count("*").alias("n"))
            .filter(col("n") > 1))
print("distinct cases with duplicate silver rows:", dups.select("CaseNo").distinct().count())
display(dups.orderBy(col("n").desc()))